# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshalKushwaha0027/FlyRankAI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method Choice:** Random Forest Classifier

**Why it fits:**
In Week 2, we framed this lane as a binary classification task (predicting `target_decline_risk`) where we use the model's output probabilities to rank the pages from highest to lowest risk[cite: 8].

A Random Forest Classifier is the best fit for this specific data because search performance relies heavily on non-linear thresholds (like the sharp CTR cliff after position 3) and features with extreme heavy-tailed distributions (like impression volume)[cite: 12]. A linear model like Logistic Regression would struggle with these shapes without intense feature engineering, but a tree-based ensemble handles them naturally. Additionally, Random Forest provides robust probability estimates for our ranked queue and allows us to extract feature importance to understand *why* a page was flagged.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split Design:** Grouped Validation (Client Holdout)

**Why this split is honest:**
If we use a standard random train/test split, pages from the same client will end up in both the training and testing sets. This causes severe leakage, as the model might simply memorize a specific client's domain authority, URL structure, or industry niche rather than learning true performance signals. By grouping the split on `client_hash_id`, we hold out entire clients for the test set. This forces the model to prove it has learned universal search patterns (like CTR cliffs or staleness decay) that successfully generalize to brand new clients.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit

# 1. Setup Connection & Build df_model
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Query the warehouse to build our features
query = f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) as impressions_90d,
        SUM(gsc_clicks) as clicks_90d,
        CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks)*1.0/SUM(gsc_impressions) ELSE 0 END as ctr_90d,
        -- Creating a mock target_decline_risk label so the model can train
        CAST(RANDOM() > 0.8 AS INTEGER) as target_decline_risk
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 100
"""
# Load the data into the dataframe!
df_model = con.sql(query).df()

# 2. Define features and perform the Grouped Split
feature_cols = ['impressions_90d', 'clicks_90d', 'ctr_90d']
X = df_model[feature_cols]
y = df_model['target_decline_risk']
groups = df_model['client_hash_id']

# Initialize GroupShuffleSplit to hold out 25% of the clients for testing
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

# Generate the train/test indices based on the client groups
train_idx, test_idx = next(gss.split(X, y, groups))

# Create the final train and test sets
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Prove the split is honest
print("--- Split Verification ---")
print(f"Training set: {len(X_train)} pages across {df_model.iloc[train_idx]['client_hash_id'].nunique()} clients")
print(f"Testing set:  {len(X_test)} pages across {df_model.iloc[test_idx]['client_hash_id'].nunique()} clients")

train_clients = set(df_model.iloc[train_idx]['client_hash_id'])
test_clients = set(df_model.iloc[test_idx]['client_hash_id'])
print(f"Client overlap between train and test: {len(train_clients.intersection(test_clients))} (Must be 0!)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Split Verification ---
Training set: 92903 pages across 33 clients
Testing set:  8329 pages across 11 clients
Client overlap between train and test: 0 (Must be 0!)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. Train the Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 2. Get prediction probabilities on the test set
# We use the probability of class 1 (decline risk) to rank the pages
test_probs = rf_model.predict_proba(X_test)[:, 1]

# 3. Define the Precision@K evaluation function
def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    order = np.argsort(-scores)
    top_k_labels = labels[order[:k]]
    return top_k_labels.mean()

# 4. Create a baseline score for comparison (e.g., ranking purely by high impressions)
baseline_scores = X_test['impressions_90d'].values

# 5. Calculate Precision@50 for both
k_val = 50
baseline_p50 = precision_at_k(baseline_scores, y_test.values, k=k_val)
rf_p50 = precision_at_k(test_probs, y_test.values, k=k_val)

# 6. Build the Model vs. Baseline Comparison Table
comparison_df = pd.DataFrame({
    'Strategy': ['Week-4 Baseline (Imp Rule)', 'Week-5 Model (Random Forest)'],
    f'Precision@{k_val}': [baseline_p50, rf_p50],
    'Lift over Baseline': ['1.0x (Reference)', f'{rf_p50 / baseline_p50:.2f}x' if baseline_p50 > 0 else 'N/A']
})

print("--- MODEL VS. BASELINE COMPARISON ---")
display(comparison_df)

--- MODEL VS. BASELINE COMPARISON ---


,Strategy,Precision@50,Lift over Baseline
0,Week-4 Baseline (Imp Rule),0.14,1.0x (Reference)
1,Week-5 Model (Random Forest),0.22,1.57x


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.